## Check if GPU is available

In [2]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
NVIDIA GeForce RTX 5070 Ti


## Resize training and validation image for RTX 5070Ti

Although YOLOv26 does not require resizing, the images are resized to the same resolution to ensure a fair comparison with RF-DETR.

In [ ]:
import json
import os
from PIL import Image

def resize_img(path, label_path, save_path, ml=1024):
    with open(label_path, "r") as f:
        coco = json.load(f)

    for img_info in coco["images"]:
        img_name = img_info["file_name"]
        img_id = img_info["id"]

        img_path = os.path.join(path, img_name)
        img = Image.open(img_path)

        old_w, old_h = img.size
        max_len = max(old_w, old_h)

        if max_len <= ml:
            img.save(os.path.join(path, img_name))
            continue

        scale = ml / max_len
        new_w = int(old_w * scale)
        new_h = int(old_h * scale)

        resized_img = img.resize((new_w, new_h), Image.LANCZOS)
        resized_img.save(os.path.join(path, img_name))

        img_info["width"] = new_w
        img_info["height"] = new_h

        for anno in coco["annotations"]:
            if anno["image_id"] == img_id:
                x, y, w, h = anno["bbox"]
                anno["bbox"] = [
                    x * scale,
                    y * scale,
                    w * scale,
                    h * scale
                ]

    with open(os.path.join(path, "_annotations.coco.json"), "w") as f:
        json.dump(coco, f, indent=4)

        
cls = ['train'] 
for c in cls:      
    path = rf'C:\rf-detr\fire_data_25\{c}'
    label_path = os.path.join(path, '_annotations.coco.json')
    save_path = os.path.join(path, '_annotations.coco.json')
    with open(label_path, 'r') as file:
        coco = json.load(file)

    max_h, max_w = 0, 0
    min_h, min_w = 0, 0

    for img in coco['images']:
        max_h = max(max_h, img['height'])
        max_w = max(max_w, img['width'])

        min_h = max(min_h, img['height'])
        min_w = max(min_w, img['width'])

    print(max_h, max_w)
    print(min_h, min_w)

    if max_h > 1024 or max_w > 1024:
        resize_img(path, label_path, save_path)



## Training

In [3]:
import torch

torch.cuda.empty_cache()

In [13]:
from ultralytics import YOLO
import torch

torch.cuda.empty_cache()

path = r"C:\ultralytics-main\examples\datasets\scoliosis.yaml"
model = YOLO('yolo26l.pt')  # load a pretrained YOLO detection model

model.train(
    data=path,
    epochs=200,
    imgsz=640,           
    lr0=0.001,
    lrf=0.01,              
    patience = 20,
    copy_paste = 0.2,
    cutmix = 0.2,
    cos_lr = True,
    cls = 1.5,
    batch = 8
)


New https://pypi.org/project/ultralytics/8.4.12 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.6  Python-3.10.15 torch-2.10.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5070 Ti, 16303MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=1.5, compile=False, conf=None, copy_paste=0.2, copy_paste_mode=flip, cos_lr=True, cutmix=0.2, data=C:\ultralytics-main\examples\datasets\scoliosis.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=200, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26l.pt, momentum=0.937, mosaic=1.0, multi_scale

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x000002368DE12AA0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.0480

In [15]:
path = r"C:\ultralytics-main\examples\datasets\scoliosis.yaml"
model_dir = r"C:\ultralytics-main\examples\runs\detect\train27\weights\best.pt"

## Validation

In [16]:
model = YOLO(model_dir)  

model.val(
    data=path,
    split="val", 
    save=True,
    conf = 0.5,
 
)

Ultralytics 8.4.6  Python-3.10.15 torch-2.10.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5070 Ti, 16303MiB)
YOLO26l summary (fused): 190 layers, 24,746,511 parameters, 0 gradients, 86.1 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 620.2455.8 MB/s, size: 67.0 KB)
val: Scanning C:\ultralytics-main\examples\datasets\scoliosis\valid\labels.cache... 100 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 100/100  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 7/7 4.4it/s 1.6s0.2s
                   all        100       1526      0.911      0.822      0.883      0.519
Speed: 0.8ms preprocess, 9.9ms inference, 0.0ms loss, 0.2ms postprocess per image
Results saved to C:\ultralytics-main\examples\runs\detect\val28


ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x0000023691AF84F0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.0480

## Testing

In [17]:
model = YOLO(model_dir)  

model.val(
    data=path,
    split="test", 
    save=True ,
    conf=0.5,
)

Ultralytics 8.4.6  Python-3.10.15 torch-2.10.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5070 Ti, 16303MiB)
YOLO26l summary (fused): 190 layers, 24,746,511 parameters, 0 gradients, 86.1 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 22.37.9 MB/s, size: 149.6 KB)
val: Scanning C:\ultralytics-main\examples\datasets\scoliosis\test\labels.cache... 100 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 100/100  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 7/7 5.0it/s 1.4s0.2s
                   all        100       1726      0.971      0.815        0.9      0.538
Speed: 0.5ms preprocess, 8.1ms inference, 0.0ms loss, 0.2ms postprocess per image
Results saved to C:\ultralytics-main\examples\runs\detect\val29


ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x000002368E758100>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.0480

## Prediction 

In [5]:
from ultralytics import YOLO

model = YOLO(r"C:\ultralytics-main\examples\runs\detect\vertebra\weights\best.pt")

model.predict(
    source=r"C:\ultralytics-main\examples\datasets\scoliosis\test\images",
    save=True,
    conf=0.2,
)



image 1/100 C:\ultralytics-main\examples\datasets\scoliosis\test\images\01-July-2019-12_jpg.rf.7b50eaa213a8b18f7265593b0a27cd6f.jpg: 640x192 19 vertebras, 88.1ms
image 2/100 C:\ultralytics-main\examples\datasets\scoliosis\test\images\01-July-2019-1_jpg.rf.a39bf99b90e14970b497076c97236a1a.jpg: 640x224 18 vertebras, 40.1ms
image 3/100 C:\ultralytics-main\examples\datasets\scoliosis\test\images\01-July-2019-25_jpg.rf.833a16bafc1a99ac9cb58aa8bac099cb.jpg: 640x192 17 vertebras, 22.9ms
image 4/100 C:\ultralytics-main\examples\datasets\scoliosis\test\images\01-July-2019-26_jpg.rf.44c817af3e3a3812fdaab0e264ea6db4.jpg: 640x256 18 vertebras, 41.8ms
image 5/100 C:\ultralytics-main\examples\datasets\scoliosis\test\images\01-July-2019-2_jpg.rf.911c98feb1c48f615f4ebfbbaedd90ef.jpg: 640x192 19 vertebras, 22.6ms
image 6/100 C:\ultralytics-main\examples\datasets\scoliosis\test\images\01-July-2019-37_jpg.rf.34338537c16b0de619813b75f780f926.jpg: 640x256 18 vertebras, 20.4ms
image 7/100 C:\ultralytics-ma

[ultralytics.engine.results.Results object with attributes:
 
 boxes: ultralytics.engine.results.Boxes object
 keypoints: None
 masks: None
 names: {0: 'vertebra'}
 obb: None
 orig_img: array([[[0, 0, 0],
         [0, 0, 0],
         [0, 0, 0],
         ...,
         [0, 0, 0],
         [0, 0, 0],
         [0, 0, 0]],
 
        [[0, 0, 0],
         [0, 0, 0],
         [0, 0, 0],
         ...,
         [0, 0, 0],
         [0, 0, 0],
         [0, 0, 0]],
 
        [[0, 0, 0],
         [0, 0, 0],
         [0, 0, 0],
         ...,
         [0, 0, 0],
         [0, 0, 0],
         [0, 0, 0]],
 
        ...,
 
        [[0, 0, 0],
         [0, 0, 0],
         [0, 0, 0],
         ...,
         [0, 0, 0],
         [0, 0, 0],
         [0, 0, 0]],
 
        [[0, 0, 0],
         [0, 0, 0],
         [0, 0, 0],
         ...,
         [0, 0, 0],
         [0, 0, 0],
         [0, 0, 0]],
 
        [[0, 0, 0],
         [0, 0, 0],
         [0, 0, 0],
         ...,
         [0, 0, 0],
         [0, 0, 0],
 

## Inference (Video)

In [6]:
import cv2
import time
import glob
import os
import torch
from ultralytics import YOLO

model = YOLO(r"C:\ultralytics-main\examples\runs\detect\fire\weights\best.pt")

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

video_folder = r"C:\ultralytics-main\examples\datasets\videos\fire"
save_folder = r"C:\ultralytics-main\examples\videos_out"

os.makedirs(save_folder, exist_ok=True)

videos = glob.glob(os.path.join(video_folder, "*.mp4"))

latencies = []
frame_count = 0

for video_path in videos:

    video_name = os.path.basename(video_path)
    save_path = os.path.join(save_folder, f"yolo_{video_name}")

    cap = cv2.VideoCapture(video_path)

    fps = int(cap.get(cv2.CAP_PROP_FPS))
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    out = cv2.VideoWriter(save_path, fourcc, fps, (w, h))

    while cap.isOpened():

        ret, frame = cap.read()
        if not ret:
            break

        if device == "cuda":
            torch.cuda.synchronize()

        start = time.perf_counter()

        results = model(frame, conf=0.2, device=0)

        if device == "cuda":
            torch.cuda.synchronize()

        end = time.perf_counter()

        frame_count += 1

        
        latencies.append(end - start)


        annotated_frame = results[0].plot()

        out.write(annotated_frame)

    cap.release()
    out.release()


avg_latency = sum(latencies) / len(latencies)
avg_latency_ms = avg_latency * 1000
fps = 1 / avg_latency

print(f"Frames evaluated: {len(latencies)}")
print(f"Average Latency: {avg_latency_ms:.2f} ms")
print(f"Inference FPS: {fps:.2f}")

Using device: cuda

0: 384x640 (no detections), 46.8ms
Speed: 6.3ms preprocess, 46.8ms inference, 0.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 21.6ms
Speed: 1.1ms preprocess, 21.6ms inference, 0.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 25.6ms
Speed: 0.9ms preprocess, 25.6ms inference, 0.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 19.9ms
Speed: 0.9ms preprocess, 19.9ms inference, 0.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 20.1ms
Speed: 1.2ms preprocess, 20.1ms inference, 0.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 19.0ms
Speed: 0.9ms preprocess, 19.0ms inference, 0.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 17.7ms
Speed: 0.9ms preprocess, 17.7ms inference, 0.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 17.6ms
Speed: 1.9ms p